# Lecture 9: Text Representation & Embeddings — Answer Key
### NLP Course 2027 — Week 05

---
Complete answers for all four exercises in Lecture 9.

In [ ]:
import nltk, numpy as np
import matplotlib.pyplot as plt
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import reuters
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer

for pkg in ['reuters', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)
print('Ready')

---
## Exercise 9.1 — Word2Vec Domain Comparison

**Task:** Train Word2Vec on Reuters. Compare nearest neighbors for common words vs a literary corpus.

**Key concept:** Word2Vec captures *distributional semantics* — words appearing in similar contexts get similar vectors. Domain strongly shapes these contexts: 'bank' clusters with financial terms in Reuters but with river/water terms in nature text.

In [ ]:
print('Loading Reuters corpus...')
reuters_sentences = [
    word_tokenize(s.lower())
    for fid in reuters.fileids()[:500]
    for s in sent_tokenize(reuters.raw(fid))
]

reuters_model = Word2Vec(
    sentences=reuters_sentences,
    vector_size=100, window=5, min_count=3,
    workers=4, epochs=10, seed=42
)
print(f'Reuters vocab size: {len(reuters_model.wv):,}')

# Compare nearest neighbors
test_words = ['bank', 'trade', 'market', 'model']
print()
for word in test_words:
    if word in reuters_model.wv:
        neighbors = [w for w, _ in reuters_model.wv.most_similar(word, topn=5)]
        print(f'Reuters  "{word}": {neighbors}')

print()
print('Observation: In Reuters, "bank" clusters with financial terms (loan, credit, deposit).')
print('In literary corpora, "bank" would cluster with river, shore, edge.')
print('Domain-specific training is critical for downstream task performance.')

---
## Exercise 9.2 — Document Similarity: Word Vectors vs TF-IDF

**Task:** Compare doc similarity via average word vectors vs TF-IDF cosine similarity.

**Key concept:** Average word vectors capture *semantic* similarity (paraphrases score high). TF-IDF captures *lexical* overlap. They disagree most when two documents share meaning but not words.

In [ ]:
def doc2vec_avg(text, model):
    """Document vector as mean of word vectors."""
    tokens = word_tokenize(text.lower())
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

doc_texts = [
    'The government raised interest rates to fight inflation.',
    'Central banks increase borrowing costs to control rising prices.',
    'Scientists discovered a new species of deep-sea fish.',
]

# Word2Vec similarity
wv_vecs = np.array([doc2vec_avg(d, reuters_model) for d in doc_texts])
wv_sim  = cosine_similarity(wv_vecs)

# TF-IDF similarity
tfidf_vec  = TfidfVectorizer().fit_transform(doc_texts)
tfidf_sim  = cosine_similarity(tfidf_vec)

labels = ['RatesDoc', 'BorrowingDoc', 'FishDoc']
print('Word2Vec cosine similarity matrix:')
print(f'{"":>12}', '  '.join(f'{l:>12}' for l in labels))
for i, row in enumerate(wv_sim):
    print(f'{labels[i]:>12}', '  '.join(f'{v:>12.4f}' for v in row))

print('\nTF-IDF cosine similarity matrix:')
print(f'{"":>12}', '  '.join(f'{l:>12}' for l in labels))
for i, row in enumerate(tfidf_sim.toarray() if hasattr(tfidf_sim, 'toarray') else tfidf_sim):
    print(f'{labels[i]:>12}', '  '.join(f'{v:>12.4f}' for v in row))

print()
print('Word2Vec should score docs 0 and 1 higher than TF-IDF (semantic paraphrase).')

---
## Exercise 9.3 — t-SNE Word Cluster Visualization

**Task:** Select ~50 words in semantic groups and visualize with t-SNE.

**Key concept:** t-SNE reduces high-dimensional embeddings to 2D while preserving local structure. Semantically related words cluster together, confirming that Word2Vec captures meaning through geometry.

In [ ]:
word_groups = {
    'finance':  ['bank', 'market', 'trade', 'price', 'stock', 'share', 'rate', 'dollar'],
    'politics': ['government', 'minister', 'president', 'policy', 'party', 'election'],
    'company':  ['company', 'corp', 'firm', 'industry', 'business', 'profit', 'loss'],
}

words_in_model, groups_in_model = [], []
for group, words in word_groups.items():
    for w in words:
        if w in reuters_model.wv:
            words_in_model.append(w)
            groups_in_model.append(group)

if len(words_in_model) >= 5:
    vectors = np.array([reuters_model.wv[w] for w in words_in_model])
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(5, len(words_in_model)-1))
    coords = tsne.fit_transform(vectors)

    colors = {'finance': 'blue', 'politics': 'red', 'company': 'green'}
    fig, ax = plt.subplots(figsize=(10, 7))
    for i, (word, group) in enumerate(zip(words_in_model, groups_in_model)):
        ax.scatter(coords[i, 0], coords[i, 1], color=colors[group], s=80)
        ax.annotate(word, (coords[i, 0], coords[i, 1]), fontsize=9)
    for group, color in colors.items():
        ax.scatter([], [], color=color, label=group)
    ax.legend()
    ax.set_title('t-SNE of Reuters Word2Vec Embeddings')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough words found in model vocabulary. Try training on more data.')

---
## Exercise 9.4 — Word Analogy Tester

**Task:** `word_analogy(model, a, b, c)` → solves b - a + c.

**Key concept:** The famous Word2Vec analogy property: king - man + woman ≈ queen. This works because gender is encoded as a consistent offset direction in the embedding space. Analogies work best on large corpora (100M+ tokens).

In [ ]:
def word_analogy(model, word_a, word_b, word_c, topn=3):
    """
    Solve: word_a is to word_b as word_c is to ?
    i.e., find words closest to: word_b - word_a + word_c
    """
    for w in [word_a, word_b, word_c]:
        if w not in model.wv:
            return f'Word "{w}" not in vocabulary'
    results = model.wv.most_similar(
        positive=[word_b, word_c],
        negative=[word_a],
        topn=topn
    )
    return [w for w, _ in results]

# Financial analogies (Reuters corpus is domain-specific)
analogies = [
    ('increase', 'rise', 'decrease'),   # -> fall
    ('buy', 'purchase', 'sell'),         # -> sale
    ('profit', 'gain', 'loss'),          # -> deficit
]
print('Word analogies on Reuters Word2Vec:')
for a, b, c in analogies:
    result = word_analogy(reuters_model, a, b, c)
    print(f'  {a} : {b} :: {c} : {result}')

print()
print('Note: "king - man + woman = queen" requires a large general corpus.')
print('On Reuters, domain analogies (buy:purchase::sell:?) work much better.')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 9.1 | Domain shapes embeddings: "bank" = financial in Reuters, geographic in literary text |
| 9.2 | Word vectors capture semantics; TF-IDF captures lexical overlap — complementary |
| 9.3 | t-SNE reveals geometric clustering of semantic groups in embedding space |
| 9.4 | Analogy = vector arithmetic; works best on large corpora, within-domain |

---
*NLP Course 2027 — Week 05*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**